Las confirmación de saldos es una técnica de auditoría que no pasa de moda, por su utilidad para obtener evidencia directamente de un tercero, quien es la fuente de información directa para los saldos bancarios, cuentas por cobrar, cuentas por pagar, entre otros aspectos legales.
Recuerdo que hace años como ayudante de auditoría tenías de facto bajo tu responsbailidad (entre otras actividades) la elaboración, envío y seguimmiento de las confirmaciones, el cual era un trabajo totalmente manual, proclive a errores de captura de datos.  

Este notebook demuestra una vez más el poder y simplicidad de python. Exceptuando las líneas necesarias para importar las librerías y comentarios, este código tiene alrededor de 15 líneas que pueden procesar cientos de confirmaciones. ¿que se necesita? necesitas una plantilla en word, para este ejemplo utilizaré este:

<img src="images/template.png" alt="template" title="plantilla ejemplo de word" width="300">

los campos sombreados (no es necesario hacerlo) y envueltos en llaves {{ }} son los datos que se alimentarán automáticamente, ya sea del script python o del archivo Excel saldos.xlsx:

<img src="images/saldos.png" alt="template" title="información a confirmar" width="400">

In [1]:
# instalar docxtpl, librería necesaria para generar documentos Word a partir de plantillas
# pip install docxtpl

In [2]:
# importar librerías necesarias
import pandas as pd
from docxtpl import DocxTemplate
from datetime import datetime
import os

In [3]:
# leer Excel con los saldos de los clientes que necesitamos confirmar
df = pd.read_excel(r'data/saldos.xlsx')

# crear carpeta de salida
os.makedirs(r'data/confirmaciones', exist_ok=True)

for _, row in df.iterrows():
    doc = DocxTemplate(r'data/template_confirmaciones.docx')
    contexto = {
        "fecha": "14/04/2025", # si es "hoy" la elaboración: datetime.today().strftime("%d/%m/%Y"),
        "fecha_corte": "31/12/2025",
        "cliente": row["cliente"],
        "moneda": row["moneda"],
        "saldo": f"{row['saldo']:,.2f}",
        "email": row['email'],
        "auditor": "LA firma de Auditoría S.C."
    }
    doc.render(contexto)

    nombre_archivo = f"data/confirmaciones/Confirmacion_{row['cliente']}.docx"
    doc.save(nombre_archivo)

    # opcional:
    print(f"Generado: {nombre_archivo}")

Generado: data/confirmaciones/Confirmacion_Comercial ABC.docx
Generado: data/confirmaciones/Confirmacion_Servicios XYZ.docx
Generado: data/confirmaciones/Confirmacion_EL Super ABC.docx


El resultado:

<img src="images/output_template.png" alt="template" title="plantilla ejemplo de word" width="300">

Listo, confirmaciones elaboradas, pero que pasa si elaboramos 20, 50, 100 confirmaciones, ¿enviaremos una por una? No, también podemos automatizar el envío. _**Aquí enfatizo que las confirmaciones ya debieran de haber sido revisadas por un auditor senior.**_

In [4]:
# instalar pywin32, librería necesaria para enviar correos electrónicos desde python en Windows
# tomar en cuenta que 1) depende del sistema operativo y 2) las directivas de seguridad de la empresa
# pip install pywin32

In [5]:
import win32com.client as win32

# usaré el archivo saldos, pero de preferencia ya debería estar elaborado el control_confirmaciones.xlsx, para seguimiento
df = pd.read_excel(r"data/saldos.xlsx")

outlook = win32.Dispatch("Outlook.Application")  # o la aplicación de correo disponible

# iteramos sobre cada fila del control de confirmaciones
for _, row in df.iterrows():
    mail = outlook.CreateItem(0)
    mail.To = row["email"]
    mail.Subject = "Solicitud de confirmación de saldo al 31/12/2025"

    mail.HTMLBody = f"""
    <p>Estimados señores de {row['cliente']}:</p>

    <p>Como parte de nuestra auditoría externa,
    agradeceremos confirmar el saldo adjunto.</p>

    <p>Saludos cordiales.</p>
    """

    archivo = os.path.abspath(
        os.path.join("data", "confirmaciones", f"Confirmacion_{row['cliente']}.docx")
    )

    if not os.path.exists(archivo):
        print(f"No existe el archivo: {archivo}")
        continue

    mail.Attachments.Add(archivo)
    mail.Send()

    print(f"Enviado a {row['cliente']}")

Enviado a Comercial ABC
Enviado a Servicios XYZ
Enviado a EL Super ABC


Ahora ve a los enviados de tu cuenta de cuenta de correo, en este caso outlook, listo se han enviado todas las confirmaciones a los clientes, con el texto tal cual lo redactaste y con el adjunto (confirmación) a prueba de errores. ¿que estamos esperando en auditoría (y en general en el ambiente de negocios) para automatizar este tipo de tareas? centremos esfuerzo en análisis, en ser estratégicos, dejemos estas tareas a scripts que automaticen el trabajo repetitivo.

<img src="images/enviado.png" alt="enviado" title="ejemplo correo enviado" width="700">